In [1]:
import requests
import xarray as xr
import gcsfs

ZARR_JSON_URL = "https://raw.githubusercontent.com/Rutgers-ESSP/IPCC-AR6-Sea-Level-Projections/main/google_zarr_stores.json"
stores = requests.get(ZARR_JSON_URL).json()

In [2]:
paths = stores["full_sample_workflows"]["gridded"]

zarr_path = [p for p in paths if "/ssp585/" in p and "/wf_1e/" in p][0]

zarr_path

'gs://ar6-lsl-simulations-public-standard/gridded/full_sample_workflows/wf_1e/ssp585/total-workflow.zarr'

In [3]:
import gcsfs
import xarray as xr

fs = gcsfs.GCSFileSystem(token="anon")

ds = xr.open_zarr(
    fs.get_mapper(zarr_path),
    consolidated=False
)

ds

<xarray.Dataset> Size: 47GB
Dimensions:           (samples: 20000, years: 9, lat: 181, lon: 360)
Coordinates:
  * samples           (samples) int64 160kB 0 1 2 3 ... 19996 19997 19998 19999
  * years             (years) int64 72B 2020 2030 2040 2050 ... 2080 2090 2100
  * lat               (lat) float64 1kB -90.0 -89.0 -88.0 ... 88.0 89.0 90.0
  * lon               (lon) float64 3kB -180.0 -179.0 -178.0 ... 178.0 179.0
Data variables:
    sea_level_change  (samples, years, lat, lon) float32 47GB dask.array<chunksize=(5000, 5, 30, 30), meta=np.ndarray>
Attributes:
    citation:              In order to document the impact of these sea-level...
    description:           Total sea-level change for workflow
    extra_info_url:        https://github.com/Rutgers-ESSP/IPCC-AR6-Sea-Level...
    history:               Created Fri Jun 18 20:50:35 2021
    license:               The IPCC AR6 Sea-Level Rise Projections are licens...
    source:                FACTS: Post-processed total among available contri...
    zarr_processing_date:  Mon Jun  6 18:16:23 2022 (UTC)

In [4]:
scenarios = ["ssp126", "ssp245", "ssp370", "ssp585"]

scenario_paths = {
    s: [p for p in paths if f"/{s}/" in p and "/wf_1e/" in p][0]
    for s in scenarios
}

scenario_paths

{'ssp126': 'gs://ar6-lsl-simulations-public-standard/gridded/full_sample_workflows/wf_1e/ssp126/total-workflow.zarr',
 'ssp245': 'gs://ar6-lsl-simulations-public-standard/gridded/full_sample_workflows/wf_1e/ssp245/total-workflow.zarr',
 'ssp370': 'gs://ar6-lsl-simulations-public-standard/gridded/full_sample_workflows/wf_1e/ssp370/total-workflow.zarr',
 'ssp585': 'gs://ar6-lsl-simulations-public-standard/gridded/full_sample_workflows/wf_1e/ssp585/total-workflow.zarr'}

In [5]:
datasets = {}

for scenario, path in scenario_paths.items():
    datasets[scenario] = xr.open_zarr(
        fs.get_mapper(path),
        consolidated=False
    )

In [6]:
ds = datasets["ssp585"]

print(ds)

<xarray.Dataset> Size: 47GB
Dimensions:           (samples: 20000, years: 9, lat: 181, lon: 360)
Coordinates:
  * samples           (samples) int64 160kB 0 1 2 3 ... 19996 19997 19998 19999
  * years             (years) int64 72B 2020 2030 2040 2050 ... 2080 2090 2100
  * lat               (lat) float64 1kB -90.0 -89.0 -88.0 ... 88.0 89.0 90.0
  * lon               (lon) float64 3kB -180.0 -179.0 -178.0 ... 178.0 179.0
Data variables:
    sea_level_change  (samples, years, lat, lon) float32 47GB dask.array<chunksize=(5000, 5, 30, 30), meta=np.ndarray>
Attributes:
    citation:              In order to document the impact of these sea-level...
    description:           Total sea-level change for workflow
    extra_info_url:        https://github.com/Rutgers-ESSP/IPCC-AR6-Sea-Level...
    history:               Created Fri Jun 18 20:50:35 2021
    license:               The IPCC AR6 Sea-Level Rise Projections are licens...
    source:                FACTS: Post-processed total among ava

In [14]:
import pandas as pd
import numpy as np

cities = {
    "Miami FL": (25.77, -80.19),
    "New Orleans LA": (29.95, -90.07),
    "Norfolk VA": (36.85, -76.29),
    "New York NY": (40.71, -74.01),
    "Atlantic City NJ": (39.36, -74.43),
    "Charleston SC": (32.78, -79.93),
    "Tampa FL": (27.95, -82.46),
    "Galveston TX": (29.30, -94.79),
    "Honolulu HI": (21.31, -157.86),
    "Seattle WA": (47.61, -122.33),
    "Boston MA": (42.36, -71.06),
    "San Francisco CA": (37.77, -122.42),
}

rows = []

for scenario, ds in datasets.items():
    for city, (lat, lon) in cities.items():

        city_ds = ds.sel(
            lat=lat,
            lon=lon,
            method="nearest"
        )

        city_ts = city_ds["sea_level_change"].median(dim="samples")

        temp = city_ts.to_dataframe().reset_index()

        temp["city"] = city
        temp["scenario"] = scenario
        temp["lat"] = lat
        temp["lon"] = lon

        rows.append(temp)

combined = pd.concat(rows, ignore_index=True)
combined.head()

,years,lat,lon,sea_level_change,city,scenario
0,2020,25.77,-80.19,75.0,Miami FL,ssp126
1,2030,25.77,-80.19,138.0,Miami FL,ssp126
2,2040,25.77,-80.19,195.0,Miami FL,ssp126
3,2050,25.77,-80.19,259.0,Miami FL,ssp126
4,2060,25.77,-80.19,315.0,Miami FL,ssp126


In [17]:
combined["sea_level_cm"] = combined["sea_level_change"] / 10
combined = combined.rename(columns={"years": "year"})

combined = combined[
    ["city", "scenario", "year", "sea_level_cm", "lat", "lon"]
]

In [18]:
combined.head()

,city,scenario,year,sea_level_cm,lat,lon
0,Miami FL,ssp126,2020,7.5,25.77,-80.19
1,Miami FL,ssp126,2030,13.8,25.77,-80.19
2,Miami FL,ssp126,2040,19.5,25.77,-80.19
3,Miami FL,ssp126,2050,25.9,25.77,-80.19
4,Miami FL,ssp126,2060,31.5,25.77,-80.19


In [19]:
combined

,city,scenario,year,sea_level_cm,lat,lon
0,Miami FL,ssp126,2020,7.500000,25.77,-80.19
1,Miami FL,ssp126,2030,13.800000,25.77,-80.19
2,Miami FL,ssp126,2040,19.500000,25.77,-80.19
3,Miami FL,ssp126,2050,25.900000,25.77,-80.19
4,Miami FL,ssp126,2060,31.500000,25.77,-80.19
...,...,...,...,...,...,...
427,San Francisco CA,ssp585,2060,26.799999,37.77,-122.42
428,San Francisco CA,ssp585,2070,34.599998,37.77,-122.42
429,San Francisco CA,ssp585,2080,43.599998,37.77,-122.42
430,San Francisco CA,ssp585,2090,54.299999,37.77,-122.42


In [20]:
interp_rows = []

for (city, scenario), group in combined.groupby(
    ["city", "scenario"]
):

    group = group.sort_values("year")

    years = pd.DataFrame({
        "year": range(
            group["year"].min(),
            group["year"].max() + 1
        )
    })

    merged = years.merge(
        group,
        on="year",
        how="left"
    )

    merged["city"] = city
    merged["scenario"] = scenario

    merged["lat"] = (
        merged["lat"]
        .ffill()
        .bfill()
    )

    merged["lon"] = (
        merged["lon"]
        .ffill()
        .bfill()
    )

    merged["sea_level_cm"] = (
        merged["sea_level_cm"]
        .interpolate(method="linear")
    )

    interp_rows.append(merged)

combined_interp = pd.concat(
    interp_rows,
    ignore_index=True
)

combined_interp.head(20)

,year,city,scenario,sea_level_cm,lat,lon
0,2020,Atlantic City NJ,ssp126,11.500000,39.36,-74.43
1,2021,Atlantic City NJ,ssp126,12.390000,39.36,-74.43
2,2022,Atlantic City NJ,ssp126,13.280000,39.36,-74.43
3,2023,Atlantic City NJ,ssp126,14.170000,39.36,-74.43
4,2024,Atlantic City NJ,ssp126,15.059999,39.36,-74.43
5,2025,Atlantic City NJ,ssp126,15.950000,39.36,-74.43
6,2026,Atlantic City NJ,ssp126,16.840000,39.36,-74.43
7,2027,Atlantic City NJ,ssp126,17.730000,39.36,-74.43
8,2028,Atlantic City NJ,ssp126,18.619999,39.36,-74.43
9,2029,Atlantic City NJ,ssp126,19.510000,39.36,-74.43


In [21]:
combined_interp.to_csv("data/all_cities_total_sea_level.csv",index=False)